# 02 — Silver Layer: Parse XML Notices (Daily)

**What this notebook does**

Reads one day of raw XML from the Bronze volume, splits notices into Contract Notices (CN) and Contract Award Notices (CAN), parses fields the dashboard and ML models need, and appends to two Delta tables in `workspace.silver`.

**Daily-append pattern**

The notebook is **idempotent per `run_date`**. Re-running it for the same date deletes that date's existing rows from Silver and re-writes them — safe for development. New days simply append.

**Parser features (kept in sync with `02b_parse_notices_backfill`):**

- ORG-ID indirection resolved for `buyer_country`, `buyer_name`, `winner_country`, `winner_name`.
- `num_tenders` computed as MEDIAN bidders-per-lot across awarded lots.
- Three fallback parsing styles (LotResult summary stats → LotTender enumeration → legacy TED_EXPORT).

**Sync rule:** If you change parsers here, mirror the change into Cell 4 / 5b of `02b_parse_notices_backfill`. The two notebooks share parser code by copy-paste.

## Cell 1 — Imports & date widget

The widget defaults to today. For backfills of a single missed day, type the date you want (YYYY-MM-DD).

In [0]:
# ── CELL 1 — Imports & date widget ───────────────────────────────────────────
import json
import re
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from datetime import datetime, date

dbutils.widgets.text("run_date", "", "Run date YYYY-MM-DD (blank = today)")
_widget = dbutils.widgets.get("run_date").strip()
run_date     = datetime.strptime(_widget, "%Y-%m-%d").date() if _widget else date.today()
run_date_str = run_date.strftime("%Y-%m-%d")

YYYY     = run_date.strftime("%Y")
MM       = run_date.strftime("%m")
YYYYMMDD = run_date.strftime("%Y%m%d")

bronze_path = f"/Volumes/workspace/default/lakehouse/bronze/ted/{YYYY}/{MM}/{YYYYMMDD}"

print(f"Run date    : {run_date_str}")
print(f"Bronze path : {bronze_path}")


## Cell 2 — Read raw XML from Bronze

`binaryFile` (not the Spark XML reader) because TED eForms uses namespace prefixes like `<urn:ContractNotice>` that confuse the XML reader. Reading as binary gives us the raw text to feed to the regex parsers.

In [0]:
# ── CELL 2 — Load Bronze XML for this run_date ───────────────────────────────
raw_df = (
    spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.xml")
        .load(bronze_path)
        .select(
            F.col("path").alias("source_file"),
            F.decode(F.col("content"), "utf-8").alias("raw_xml"),
        )
)
print(f"Raw XML files loaded: {raw_df.count():,}")


## Cell 3 — Regex helpers and the notice-type detector

All parsing is done with simple regex inside Python UDFs. Each helper returns `None` on miss so we can chain fallbacks with `or`. CDATA wrappers are stripped because some legacy notices wrap text in them.

In [0]:
# ── CELL 3 — Regex helpers ───────────────────────────────────────────────────
def extract_regex(pattern, text):
    if not text:
        return None
    m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    if not m:
        return None
    val = re.sub(r'<!\[CDATA\[(.*?)\]\]>', r'\1', m.group(1), flags=re.DOTALL)
    return val.strip() or None

def first_match(patterns, text):
    for p in patterns:
        v = extract_regex(p, text)
        if v:
            return v
    return None

def detect_notice_type(xml_str):
    if not xml_str:
        return "OTHER"
    if "<urn:ContractNotice" in xml_str or "<ContractNotice" in xml_str:
        return "CN"
    if "<urn:ContractAwardNotice" in xml_str or "<ContractAwardNotice" in xml_str:
        return "CAN"
    if "<TED_EXPORT" in xml_str:
        if "F02_2014" in xml_str:    return "CN"
        if "F03_2014" in xml_str:    return "CAN"
    return "OTHER"


## Cell 4 — Shared field extractors

Two patterns of indirection are handled here:

1. **Buyer fields** — `<cac:ContractingParty>` contains only an `ORG-ID`. The actual buyer name and country live in a matching `<efac:Organization>` block, looked up by ID.
2. **Winner fields** — `<efac:Tenderer>` (inside `<efac:LotResult>`) contains only an `ORG-ID`. Winner name/country are looked up the same way.

Both use the shared helpers `_find_org_block` + the two ORG-ID extractors.

In [0]:
# ── CELL 4 — Shared field extractors ─────────────────────────────────────────

# ─── eForms ORG-ID resolution helpers ───────────────────────────────────────
def _find_org_block(xml, target_id):
    """Return the <efac:Organization>...</efac:Organization> block whose
    <cbc:ID> equals target_id, or None. eForms-specific."""
    if not (xml and target_id):
        return None
    for block in re.findall(r'<efac:Organization>(.*?)</efac:Organization>', xml, re.DOTALL):
        m = re.search(r'<cbc:ID[^>]*>(.*?)</cbc:ID>', block)
        if m and m.group(1).strip() == target_id:
            return block
    return None

def _contracting_party_org_id(xml):
    """ORG-ID referenced by <cac:ContractingParty>, e.g. 'ORG-0002'."""
    m = re.search(
        r'<cac:ContractingParty>.*?<cac:PartyIdentification>\s*<cbc:ID[^>]*>(.*?)</cbc:ID>',
        xml or "", re.DOTALL,
    )
    return m.group(1).strip() if m else None

def _tenderer_org_id(xml):
    """ORG-ID of the winning tenderer. Prefer the reference inside
    <efac:LotResult> (the winner), fall back to any <efac:Tenderer> block."""
    m = re.search(
        r'<efac:LotResult>.*?<efac:Tenderer>\s*<cbc:ID[^>]*>(.*?)</cbc:ID>',
        xml or "", re.DOTALL,
    )
    if m:
        return m.group(1).strip()
    m = re.search(
        r'<efac:Tenderer>\s*<cbc:ID[^>]*>(.*?)</cbc:ID>',
        xml or "", re.DOTALL,
    )
    return m.group(1).strip() if m else None

# ─── Common metadata fields ──────────────────────────────────────────────────
def x_notice_id(xml):
    return first_match([
        r'<cbc:ID[^>]*schemeName="notice-id"[^>]*>(.*?)</cbc:ID>',
        r'<NO_DOC_OJS>(.*?)</NO_DOC_OJS>',
    ], xml)

def x_publication_date(xml):
    return first_match([
        r'<cbc:IssueDate[^>]*>(.*?)</cbc:IssueDate>',
        r'<efbc:TransmissionDate[^>]*>(.*?)</efbc:TransmissionDate>',
        r'<DATE_PUB>(.*?)</DATE_PUB>',
    ], xml)

# ─── Buyer fields (resolved via ContractingParty ORG-ID) ─────────────────────
def x_buyer_name(xml):
    """Buyer organization name. Resolves ContractingParty ORG-ID."""
    org = _find_org_block(xml, _contracting_party_org_id(xml))
    if org:
        m = re.search(r'<cac:PartyName>\s*<cbc:Name[^>]*>(.*?)</cbc:Name>', org, re.DOTALL)
        if m:
            return m.group(1).strip()
    return first_match([
        r'<cac:ContractingParty>.*?<cbc:Name[^>]*>(.*?)</cbc:Name>',
        r'<OFFICIALNAME>(.*?)</OFFICIALNAME>',
    ], xml)

def x_buyer_country(xml):
    """Buyer ISO country code. Resolves ContractingParty ORG-ID."""
    org = _find_org_block(xml, _contracting_party_org_id(xml))
    if org:
        m = re.search(
            r'<cac:Country>\s*<cbc:IdentificationCode[^>]*>(.*?)</cbc:IdentificationCode>',
            org, re.DOTALL,
        )
        if m:
            return m.group(1).strip()
    return first_match([
        r'<cac:ContractingParty>.*?<cac:Country>\s*<cbc:IdentificationCode[^>]*>(.*?)</cbc:IdentificationCode>',
        r'<cac:Party>.*?<cac:Country>\s*<cbc:IdentificationCode[^>]*>(.*?)</cbc:IdentificationCode>',
        r'<ISO_COUNTRY VALUE="([A-Z]{2,3})"',
    ], xml)

def x_buyer_type(xml):
    return first_match([
        r'<cbc:PartyTypeCode[^>]*listName="buyer-legal-type"[^>]*>(.*?)</cbc:PartyTypeCode>',
        r'<cbc:ContractingPartyTypeCode[^>]*>(.*?)</cbc:ContractingPartyTypeCode>',
        r'<CA_TYPE[^>]*VALUE="([^"]+)"',
    ], xml)
def x_notice_subtype(xml):
    """eForms notice subtype code. 29-32 = standard CAN awards;
    38 = contract modification; 33 = ex-ante; E2-E4 = below-threshold national.
    Used to whitelist real awards downstream. CAN-only field."""
    return first_match([
        r'<cbc:SubTypeCode[^>]*listName="notice-subtype"[^>]*>(.*?)</cbc:SubTypeCode>',
        r'<cbc:SubTypeCode[^>]*>(.*?)</cbc:SubTypeCode>',
    ], xml)

# ─── Winner fields (resolved via Tenderer ORG-ID) ────────────────────────────
def x_winner_name(xml):
    """Winner organization name. Resolves Tenderer ORG-ID — without this
    the parser would greedy-match across blocks and return the buyer's name
    instead of the winner's. Critical for the is_cross_border ML feature."""
    org = _find_org_block(xml, _tenderer_org_id(xml))
    if org:
        m = re.search(r'<cac:PartyName>\s*<cbc:Name[^>]*>(.*?)</cbc:Name>', org, re.DOTALL)
        if m:
            return m.group(1).strip()
    return first_match([
        r'<CONTRACTOR>.*?<OFFICIALNAME>(.*?)</OFFICIALNAME>',
    ], xml)

def x_winner_country(xml):
    """Winner ISO country code. Resolves Tenderer ORG-ID. Needed for
    gold.ml_features.is_cross_border = (buyer_country != winner_country)."""
    org = _find_org_block(xml, _tenderer_org_id(xml))
    if org:
        m = re.search(
            r'<cac:Country>\s*<cbc:IdentificationCode[^>]*>(.*?)</cbc:IdentificationCode>',
            org, re.DOTALL,
        )
        if m:
            return m.group(1).strip()
    return first_match([
        r'<CONTRACTOR>.*?<ISO_COUNTRY VALUE="([A-Z]{2,3})"',
    ], xml)

# ─── Procurement fields ──────────────────────────────────────────────────────
def x_project_title(xml):
    return first_match([
        r'<cac:ProcurementProject>.*?<cbc:Name[^>]*>(.*?)</cbc:Name>',
        r'<cbc:Title[^>]*>(.*?)</cbc:Title>',
        r'<TITLE>(.*?)</TITLE>',
    ], xml)

def x_main_cpv(xml):
    return first_match([
        r'<cbc:ItemClassificationCode[^>]*listName="cpv"[^>]*>(.*?)</cbc:ItemClassificationCode>',
        r'<cbc:ItemClassificationCode[^>]*>(.*?)</cbc:ItemClassificationCode>',
        r'<CPV_CODE CODE="([^"]+)"',
    ], xml)

def x_procedure_type(xml):
    return first_match([
        r'<cbc:ProcedureCode[^>]*>(.*?)</cbc:ProcedureCode>',
        r'<cbc:TenderingProcessReason[^>]*>(.*?)</cbc:TenderingProcessReason>',
    ], xml)

def x_currency_from(xml, value_tag):
    return extract_regex(rf'<{value_tag}[^>]*currencyID="([^"]+)"', xml)

# ─── Bidder count (CAN-only, multi-style fallback) ───────────────────────────
def x_num_tenders(xml):
    """
    Bidder count per CAN, as the MEDIAN bids-per-lot across lots that
    received at least one bid. Median (not min/max/sum) gives the typical
    lot's competition level — robust to long-tail framework agreements
    where some lots routinely attract 1 bidder while most attract many.

    Three parsing styles tried in order:
      1. <efac:LotResult> summary stats (modern eForms with aggregated counts).
      2. <efac:LotTender> enumeration grouped by lot (modern eForms with
         per-bid detail). Bidders-per-lot inferred from the count of
         LotTender blocks referencing each lot ID.
      3. Legacy <ReceivedTendersQuantity> / <NB_TENDERS_RECEIVED> tags
         for pre-eForms XML.
    """
    if not xml:
        return None

    from statistics import median

    # Style 1: <efac:LotResult> summary stats.
    pairs = re.findall(
        r'<efbc:StatisticsCode[^>]*>(.*?)</efbc:StatisticsCode>\s*'
        r'<efbc:StatisticsNumeric[^>]*>(\d+)</efbc:StatisticsNumeric>',
        xml, re.DOTALL,
    )
    per_lot = [int(n) for code, n in pairs if code.strip() == "tenders" and int(n) > 0]
    if per_lot:
        return str(int(median(per_lot)))

    # Style 2: count <efac:LotTender> blocks grouped by <efac:TenderLot> ID.
    from collections import Counter
    lot_counts = Counter()
    for block in re.findall(r'<efac:LotTender>(.*?)</efac:LotTender>', xml, re.DOTALL):
        m = re.search(r'<efac:TenderLot>\s*<cbc:ID[^>]*>(.*?)</cbc:ID>', block, re.DOTALL)
        if m:
            lot_counts[m.group(1).strip()] += 1
    if lot_counts:
        return str(int(median(lot_counts.values())))

    # Style 3: legacy pre-eForms fallback.
    return first_match([
        r'<cbc:ReceivedTendersQuantity[^>]*>(.*?)</cbc:ReceivedTendersQuantity>',
        r'<NB_TENDERS_RECEIVED[^>]*>(.*?)</NB_TENDERS_RECEIVED>',
    ], xml)


## Cell 5 — CN and CAN parsers

In [0]:
# ── CELL 5a — CN parser ──────────────────────────────────────────────────────
def parse_cn(xml):
    try:
        return json.dumps({
            "notice_id":          x_notice_id(xml),
            "publication_date":   x_publication_date(xml),
            "buyer_name":         x_buyer_name(xml),
            "buyer_country":      x_buyer_country(xml),
            "buyer_type":         x_buyer_type(xml),
            "project_title":      x_project_title(xml),
            "description":        first_match([
                r'<cbc:Description[^>]*>(.*?)</cbc:Description>',
                r'<SHORT_DESCR>(.*?)</SHORT_DESCR>',
            ], xml),
            "cpv_code":           x_main_cpv(xml),
            "procedure_type":     x_procedure_type(xml),
            "estimated_value":    first_match([
                r'<cbc:EstimatedOverallContractAmount[^>]*>(.*?)</cbc:EstimatedOverallContractAmount>',
                r'<VAL_ESTIMATED_TOTAL[^>]*>(.*?)</VAL_ESTIMATED_TOTAL>',
            ], xml),
            "currency":           x_currency_from(xml, "cbc:EstimatedOverallContractAmount"),
            "num_lots":           len(re.findall(r'<cac:ProcurementProjectLot[\s>]', xml or "")) or None,
            "submission_deadline": first_match([
                r'<cbc:EndDate[^>]*>(.*?)</cbc:EndDate>',
                r'<DT_DATE_FOR_SUBMISSION[^>]*>(.*?)</DT_DATE_FOR_SUBMISSION>',
            ], xml),
            "parse_error":        None,
        })
    except Exception as e:
        return json.dumps({"parse_error": str(e)})


In [0]:
# ── CELL 5b — CAN parser ─────────────────────────────────────────────────────
def parse_can(xml):
    try:
        award_value = first_match([
            r'<cbc:TaxExclusiveAmount[^>]*>(.*?)</cbc:TaxExclusiveAmount>',
            r'<cbc:PayableAmount[^>]*>(.*?)</cbc:PayableAmount>',
            r'<VAL_TOTAL[^>]*>(.*?)</VAL_TOTAL>',
        ], xml)
        currency = (
            x_currency_from(xml, "cbc:TaxExclusiveAmount")
            or x_currency_from(xml, "cbc:PayableAmount")
        )
        return json.dumps({
            "notice_id":         x_notice_id(xml),
            "notice_subtype":    x_notice_subtype(xml),
            "publication_date":  x_publication_date(xml),
            "buyer_name":        x_buyer_name(xml),
            "buyer_country":     x_buyer_country(xml),
            "buyer_type":        x_buyer_type(xml),
            "project_title":     x_project_title(xml),
            "cpv_code":          x_main_cpv(xml),
            "procedure_type":    x_procedure_type(xml),
            "result_code":       first_match([
                r'<cbc:TenderResultCode[^>]*>(.*?)</cbc:TenderResultCode>',
                r'<efbc:TenderResultCode[^>]*>(.*?)</efbc:TenderResultCode>',
            ], xml),
            "award_value":       award_value,
            "currency":          currency,
            "winner_name":       x_winner_name(xml),
            "winner_country":    x_winner_country(xml),
            "num_tenders":       x_num_tenders(xml),
            "parse_error":       None,
        })
    except Exception as e:
        return json.dumps({"parse_error": str(e)})


In [0]:
# ── CELL 5c — Register UDFs ──────────────────────────────────────────────────
detect_type_udf = F.udf(detect_notice_type, StringType())
parse_cn_udf    = F.udf(parse_cn,           StringType())
parse_can_udf   = F.udf(parse_can,          StringType())


## Cell 6 — Split notices by type

In [0]:
# ── CELL 6 — Split into CN and CAN ───────────────────────────────────────────
typed_df   = raw_df.withColumn("notice_type", detect_type_udf(F.col("raw_xml")))
cn_raw_df  = typed_df.filter(F.col("notice_type") == "CN")
can_raw_df = typed_df.filter(F.col("notice_type") == "CAN")

typed_df.groupBy("notice_type").count().show()


## Cell 7 — Apply parsers

`from_json` decodes the parser's JSON output into the typed schema below. Adding a column later means appending one `StructField` here and one line in the parser — nothing else changes.

In [0]:
# ── CELL 7a — Apply CN parser ────────────────────────────────────────────────
cn_schema = StructType([
    StructField("notice_id",           StringType(),  True),
    StructField("publication_date",    StringType(),  True),
    StructField("buyer_name",          StringType(),  True),
    StructField("buyer_country",       StringType(),  True),
    StructField("buyer_type",          StringType(),  True),
    StructField("project_title",       StringType(),  True),
    StructField("description",         StringType(),  True),
    StructField("cpv_code",            StringType(),  True),
    StructField("procedure_type",      StringType(),  True),
    StructField("estimated_value",     StringType(),  True),
    StructField("currency",            StringType(),  True),
    StructField("num_lots",            IntegerType(), True),
    StructField("submission_deadline", StringType(),  True),
    StructField("parse_error",         StringType(),  True),
])

cn_parsed_df = (
    cn_raw_df
      .withColumn("parsed", F.from_json(parse_cn_udf(F.col("raw_xml")), cn_schema))
      .select(
          "parsed.*",
          F.col("source_file"),
          F.current_timestamp().alias("parsed_at"),
          F.lit(run_date_str).cast("date").alias("run_date"),
      )
)


In [0]:
# ── CELL 7b — Apply CAN parser ───────────────────────────────────────────────
can_schema = StructType([
    StructField("notice_id",         StringType(), True),
    StructField("notice_subtype",    StringType(), True),
    StructField("publication_date",  StringType(), True),
    StructField("buyer_name",        StringType(), True),
    StructField("buyer_country",     StringType(), True),
    StructField("buyer_type",        StringType(), True),
    StructField("project_title",     StringType(), True),
    StructField("cpv_code",          StringType(), True),
    StructField("procedure_type",    StringType(), True),
    StructField("result_code",       StringType(), True),
    StructField("award_value",       StringType(), True),
    StructField("currency",          StringType(), True),
    StructField("winner_name",       StringType(), True),
    StructField("winner_country",    StringType(), True),
    StructField("num_tenders",       StringType(), True),
    StructField("parse_error",       StringType(), True),
])

can_parsed_df = (
    can_raw_df
      .withColumn("parsed", F.from_json(parse_can_udf(F.col("raw_xml")), can_schema))
      .select(
          "parsed.*",
          F.col("source_file"),
          F.current_timestamp().alias("parsed_at"),
          F.lit(run_date_str).cast("date").alias("run_date"),
      )
)


## Cell 8 — Coverage check

Quick sanity check on the columns that drive the ML models. Cast to string before the empty-check so numeric columns (e.g. `num_lots`) don't trigger CAST_INVALID_INPUT.

In [0]:
# ── CELL 8 — Field coverage ──────────────────────────────────────────────────
def coverage(df, cols):
    """Fraction of rows where the column is non-null and non-empty."""
    total = df.count()
    if total == 0:
        print("  (empty)")
        return
    return df.select([
        F.round(
            F.count(F.when(F.col(c).isNotNull() & (F.col(c).cast("string") != ""), c)) / F.lit(total),
            3,
        ).alias(c)
        for c in cols
    ])

print("CN field coverage:")
coverage(cn_parsed_df,
         ["notice_id", "buyer_country", "buyer_name", "buyer_type", "cpv_code",
          "procedure_type", "estimated_value", "num_lots"]).show()

print("CAN field coverage:")
coverage(can_parsed_df,
         ["notice_id", "buyer_country", "buyer_name", "buyer_type", "cpv_code",
          "procedure_type", "result_code", "award_value",
          "winner_name", "winner_country", "num_tenders"]).show()


## Cell 9 — Write CN to Delta (idempotent append by `run_date`)

Delete this run_date's existing rows first, then append. Re-runs are safe; the table grows one day at a time.

In [0]:
# ── CELL 9 — Write CN to Delta ───────────────────────────────────────────────
try:
    spark.sql(
        f"DELETE FROM workspace.silver.contract_notices WHERE run_date = '{run_date_str}'"
    )
    print(f"Cleared existing CN rows for {run_date_str}.")
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("First run — table does not exist yet.")
    else:
        raise

(cn_parsed_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.silver.contract_notices"))

print(f"Appended CN rows for {run_date_str}.")


## Cell 10 — Write CAN to Delta (idempotent append by `run_date`)

In [0]:
# ── CELL 10 — Write CAN to Delta ─────────────────────────────────────────────
try:
    spark.sql(
        f"DELETE FROM workspace.silver.contract_award_notices_subtype WHERE run_date = '{run_date_str}'"
    )
    print(f"Cleared existing CAN rows for {run_date_str}.")
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("First run — table does not exist yet.")
    else:
        raise

(can_parsed_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.silver.contract_award_notices_subtype"))

print(f"Appended CAN rows for {run_date_str}.")


## Cell 11 — Verify

Counts by `run_date` — the table should grow by one entry per day. If a date appears twice, the DELETE in Cell 9/10 didn't run.

In [0]:
# ── CELL 11 — Verify Silver tables ───────────────────────────────────────────
print("--- Contract Notices by run_date ---")
spark.sql("""
    SELECT run_date, COUNT(*) AS rows
    FROM workspace.silver.contract_notices
    GROUP BY run_date ORDER BY run_date DESC
""").show()

print("--- Contract Award Notices by run_date ---")
spark.sql("""
    SELECT run_date, COUNT(*) AS rows
    FROM workspace.silver.contract_award_notices_subtype
    GROUP BY run_date ORDER BY run_date DESC
""").show()
